<a href="https://colab.research.google.com/github/musab855/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

An **audit** notebook, not a modeling notebook. Two moves, in order:

1. **Read the FlyRank research paper the way I would want my own work read** — two findings, and for each one the methodology question I would ask: *where does the label come from? does the validation design carry the claim?* Framed constructively; the goal is the next level of rigor, not a grade.
2. **Turn the same lens on my Week-5 model** — re-run it under an honest split with a before/after, hunt leakage on the final feature set, show real failure examples, and rewrite any of my own claims that run ahead of the evidence.

**Skills loaded for this task:** `hunting-leakage-and-validating` (+ `flyrank/flyrank-data` for the data).
**Claim vocabulary for every sentence below:** *observed, measured, directional, decision-support.*

Paper: `docs/flyrank-seo-research-march-2026.pdf` (FlyRank, "The State of AI-Driven SEO", March 2026).
Model under audit: `work/notebooks/w05_model.ipynb` (ML-08). No client names, URLs, or raw queries anywhere; the scan cache lives under `work/outputs/` and is gitignored.

In [1]:
# ---- Setup -----------------------------------------------------------------
# Runs from the repo root (Colab badge clone or local clone). The token is read
# from Colab Secrets (same as Weeks 4 and 5), the environment, or a local .env
# file and is NEVER printed, stored, or written anywhere.
import os, sys, getpass, subprocess, importlib.util, shutil
from pathlib import Path

def find_repo_root(start):
    for p in [start, *start.parents]:
        if (p / "work").is_dir() and (p / "skills").is_dir():
            return p
    return None

root = find_repo_root(Path.cwd())
if root is None:
    # Not inside the repo yet. Reuse an existing clone if there is one
    # (re-running this cell, or an earlier notebook in the same runtime).
    if Path("/content").exists():
        dest = Path("/content/flyrank-ml-internship")   # Colab
    else:
        dest = Path.cwd() / "flyrank-ml-internship"     # local / other
    if (dest / "work").is_dir() and (dest / "skills").is_dir():
        print(f"repo folder already present - reusing it: {dest}")
        root = dest
    else:
        if dest.exists():
            # leftover from an interrupted clone - start clean
            shutil.rmtree(dest)
        proc = subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/musab855/flyrank-ml-internship.git", str(dest)],
            capture_output=True, text=True)
        if proc.returncode != 0:
            print(proc.stdout)
            print(proc.stderr)
            raise RuntimeError(
                f"git clone failed (exit {proc.returncode}) - "
                "read git's message above, fix it, then re-run this cell.")
        root = dest
os.chdir(root)

missing = [p for p in ("duckdb", "huggingface_hub", "pypdf")
           if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    importlib.invalidate_caches()

def load_token():
    # Colab Secrets first (Week 4 stored it as HF_TOKEN2, then HF_TOKEN).
    try:
        from google.colab import userdata
        for key in ("HF_TOKEN2", "HF_TOKEN"):
            try:
                token = userdata.get(key)
                if token:
                    return token
            except Exception:
                pass
    except Exception:
        pass
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    env_file = Path(".env")
    if env_file.exists():
        for line in env_file.read_text().splitlines():
            if line.strip().startswith("HF_TOKEN"):
                return line.split("=", 1)[1].strip().strip(chr(34) + chr(39))
    return getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

print("working directory:", Path.cwd())

repo folder already present - reusing it: /content/flyrank-ml-internship
working directory: /content/flyrank-ml-internship


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [2]:
# ---- The two findings, quoted verbatim from the paper ----------------------
# I extract the exact sentences from the PDF rather than paraphrasing from
# memory, so the questions below are aimed at what the paper actually says.
from pypdf import PdfReader

PDF = Path("docs/flyrank-seo-research-march-2026.pdf")
pdf = PdfReader(str(PDF))
print(f"{PDF.name}: {len(pdf.pages)} pages")

NEEDLES = [
    "37.6% longer",          # Finding #1, CONFIRMED — page 6
    "71% holdout accuracy",  # ML appendix, logistic regression — page 29
]
found = set()
for i, page in enumerate(pdf.pages):
    text = " ".join((page.extract_text() or "").split())
    for n in NEEDLES:
        j = text.find(n)
        if j >= 0:
            found.add(n)
            print(f"\n--- verbatim | page {i+1} | needle: {n!r} ---")
            print(text[max(0, j - 300): j + 340])

assert found == set(NEEDLES), f"could not find: {set(NEEDLES) - found}"
print("\nBoth quotes located in the PDF — the questions below quote the paper, not a memory of it.")

flyrank-seo-research-march-2026.pdf: 36 pages

--- verbatim | page 6 | needle: '37.6% longer' ---
r, younger, and already positioned slightly better in search. 3180 avg words (growing) 2311 avg words (declining) 184d avg age (growing) 230d avg age (declining) DIRECTION COUNT WORDS AGE AVG IMP AVG POS HEALTH up 74.8K 3.2K 184d 2.8K 15.9 36.4 down 45.6K 2.3K 230d 2.6K 16.1 32.4 Growing content is 37.6% longer (3.2K vs 2.3K words) and 20% younger (184 vs 230 days). Interestingly, declining content is not simply "bad content" in the abstract. Many declining pages still carry meaningful impressions, but they are older, thinner, and less likely to retain momentum. The sample sizes here are large (74,187 rising vs 45,272 falling), so t

--- verbatim | page 29 | needle: '71% holdout accuracy' ---
FlyRank ML APPENDIX — GROWTH & CLASSIFICATION What Predicts Growth? Logistic regression (71% holdout accuracy) describing which sampled features separate growing from declining pages. GROWTH PREDICTIO

### Finding #1 — CONFIRMED: "The Anatomy of Growing Content" (page 6)

**What the paper reports** (verbatim, printed by the cell above):

> "Growing content is 37.6% longer (3.2K vs 2.3K words) and 20% younger (184 vs 230 days). … The sample sizes here are large (74,187 rising vs 45,272 falling), so the word-count and age gap is directionally robust even though this remains an observational comparison."

The paper already tags this observational and discloses its own standard (page 4: headline findings prioritize direct aggregate comparisons; page 36: no p-values or confidence intervals are reported). That is exactly the standard I want my work held to — so here are the two questions I would still ask, and would want asked about my own numbers:

**Q1 — Where does the label come from?**
The two cohorts are defined by `trend_direction` (page 5: "Up: >10% growth. Down: >10% decline", computed from 30d-vs-prev-30d impressions). So the cohort membership is one ratio crossing a ±10% threshold. Questions: how many pages in each cohort sit *near* that edge, where a handful of impressions flips them from "up" to "down"? How many sit at volumes where 2 → 3 impressions still counts as growth? The paper's n ≥ 50 rule is stated for analysis buckets — is it also applied to the cohort definition itself?
*How to make it stronger:* report each cohort's impressions floor and the share of pages within ±10% of the threshold, or hold the borderline pages out as their own cohort and show whether the 37.6% / 20% gaps survive.

**Q2 — Does the validation design carry the claim?**
The rising and falling cohorts differ in more than word count: client mix, content type, and season all move with a trend label at the same time. A large `n` shrinks noise; it does not shrink confounding. The finding itself stays observational — the one step I would flag sits in the recommendation that follows: "Expand thin pages … Expected: Improves the odds that an already visible page can keep growing" reads one rung above what a cross-sectional association supports.
*How to make it stronger:* compare within client (each client acting as its own control), or match pages on age and visibility before comparing word counts — or measure refreshed vs. not-refreshed pages before and after, at which point "Expected" can become "measured". Until then the honest form is decision-support: *these pages look worth reviewing first.*

Framed this way because this is precisely the review I would want on Week 5: two specific questions, each with a concrete way to make the result stronger.

### Finding #2 — ML appendix: "What Predicts Growth?" (page 29): logistic regression, 71% holdout accuracy

**What the paper reports** (verbatim):

> "Logistic regression (71% holdout accuracy) describing which sampled features separate growing from declining pages."

The appendix is explicitly labeled exploratory and secondary to the direct evidence (pages 4 and 36) — the questions below are about making the number stronger, not about grading it.

**Q1 — Where does the label come from, and what does a naive guess score?**
"growing vs declining" appears to be the same ±10% `trend_direction` label used elsewhere in the paper; page 29 does not restate the definition. The more concrete gap: **71% accuracy is printed with no base rate beside it.** If roughly two-thirds of the 61.8K sample is "growing", then always answering "growing" already lands near that number, and 71% would be a few points of measured skill rather than a headline.
*How to make it stronger:* print majority-class accuracy next to 71% (my tables in section 2 print the 0.465 base rate under every metric for exactly this reason), and/or report a ranking metric — ROC-AUC or average precision — that does not depend on one threshold.

**Q2 — Does the validation design support the claim?**
Page 36 describes an 80/20 split but does not say the split was **grouped by brand**. With 57 brands, a row-random 80/20 puts pages from the same brand on both sides; brands share templates, seasonality, and measurement quirks, so a model can score well by recognizing brand fingerprints rather than by learning growth. The question is the one the leakage skill asks of everyone: *would 71% survive holding out whole brands?*
*How to make it stronger:* grouped (brand-held-out) evaluation, plus the before/after that shows what the row-random split was worth. That is exactly the experiment I run on my own model in the next section — same question, turned on myself.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**What runs in this section:** the same March 2026 frame, the same three features, the same two models, the same seed (42) and the same metrics as Week 5 — scored under **two different splits**:

- **BEFORE — row-random 5-fold.** Rows shuffled, folds cut by row. Pages from the same client land on both sides of every fold. This is the default many notebooks ship, and it is the number that flatters the model, because the forest gets to train on pages from clients it is also tested on.
- **AFTER — client-grouped 5-fold.** Every client sits in exactly one test fold; no client appears in train and test at once. The honest question: *does it work on a client the model has never seen?* This is deliberately the split Week 5 reported, so the AFTER table doubles as a reproduction check of my own published numbers.

Metrics: **P@10 / P@20 / P@50, ROC-AUC, average precision**, and the **base rate printed under every table** — a score without its naive baseline is half a number. The three non-learned rows (`rule`, `rule_full`, `impressions_only`) have no training step, so their numbers must be *identical* under both splits; if they move, the harness is broken and nothing below it can be trusted.

In [3]:
# ---- Config ----------------------------------------------------------------
import json
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

RANDOM_STATE = 42           # every random step in this notebook uses this one seed
N_FOLDS = 5
BASE_FEATURES = ["gsc_impressions_avg", "gsc_clicks_avg", "position_avg"]
AUDIT_ONLY = ["gsc_impressions_16to31"]   # label-window column: audit demo only, NEVER a feature
IMPRESSIONS_THRESHOLD = 500  # the Week-4 rule's cutoff, unchanged
POSITION_THRESHOLD = 11      # the Week-4 rule's cutoff, unchanged
METRIC_ORDER = ["precision_at_10", "precision_at_20", "precision_at_50", "roc_auc", "avg_precision"]

print(f"numpy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")
print("random_state:", RANDOM_STATE, "- fixed, so a rerun reproduces every table below")
print("features:", BASE_FEATURES)
print("audit-only column (used once, in section 3):", AUDIT_ONLY)

numpy 2.1.3 | pandas 2.2.3 | scikit-learn 1.6.1
random_state: 42 - fixed, so a rerun reproduces every table below
features: ['gsc_impressions_avg', 'gsc_clicks_avg', 'position_avg']
audit-only column (used once, in section 3): ['gsc_impressions_16to31']


In [4]:
# ---- Data: same SQL, same window, same filters as Week 4 --------------------
# March 2026 only. Features come from days 1-15; the label is defined on days 16-31.
# This scan adds the day-16-31 impressions column to the cache: it is the label's
# own window, kept ONLY so section 3 can prove the harness detects it as leakage.
# The heavy scan runs once and is cached (the cache file is gitignored, never committed).
CACHE = Path("work/outputs/feature_frame_march_audit.csv")

if CACHE.exists():
    df = pd.read_csv(CACHE)
    print(f"Cache hit: {len(df):,} rows from {CACHE}")
else:
    HF_TOKEN = load_token()
    import duckdb
    con = duckdb.connect()
    con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
    rel = "hf://datasets/FlyRank/internship-warehouse"

    df = con.sql(f'''
    WITH daily_averages AS (
      SELECT
        content_hash_id,
        client_hash_id,
        AVG(CASE WHEN report_date < '2026-03-16' AND gsc_data_available THEN gsc_impressions ELSE NULL END) as gsc_impressions_1to15,
        AVG(CASE WHEN report_date < '2026-03-16' AND gsc_data_available THEN gsc_clicks ELSE NULL END) as gsc_clicks_1to15,
        AVG(CASE WHEN report_date < '2026-03-16' AND gsc_data_available THEN gsc_avg_position ELSE NULL END) as position_1to15,
        AVG(CASE WHEN report_date >= '2026-03-16' AND gsc_data_available THEN gsc_impressions ELSE NULL END) as gsc_impressions_16to31
      FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
      WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01' AND gsc_data_available = TRUE
      GROUP BY content_hash_id, client_hash_id
    )
    SELECT
      content_hash_id AS content_id,
      client_hash_id,
      gsc_impressions_1to15 as gsc_impressions_avg,
      gsc_clicks_1to15 as gsc_clicks_avg,
      position_1to15 as position_avg,
      gsc_impressions_16to31,
      (CASE
         WHEN gsc_impressions_16to31 IS NULL OR gsc_impressions_1to15 IS NULL THEN NULL
         WHEN gsc_impressions_16to31 < gsc_impressions_1to15 THEN 1
         ELSE 0
       END) as is_declining_label
    FROM daily_averages
    ''').df()

    CACHE.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(CACHE, index=False)
    print(f"Scanned the warehouse (March 2026): {len(df):,} rows -> cached at {CACHE}")

print("cache columns:", list(df.columns))

Cache hit: 176,738 rows from work/outputs/feature_frame_march_audit.csv
cache columns: ['content_id', 'client_hash_id', 'gsc_impressions_avg', 'gsc_clicks_avg', 'position_avg', 'gsc_impressions_16to31', 'is_declining_label']


In [5]:
# Same filters as Week 4: undefined label or no position data -> dropped, not guessed.
n_before = len(df)
pos_nan  = df["position_avg"].isna()
pos_zero = df["position_avg"] == 0
lab_nan  = df["is_declining_label"].isna()

n_pos_nan, n_pos_zero = int(pos_nan.sum()), int(pos_zero.sum())
n_lab_nan_extra = int((lab_nan & ~pos_nan & ~pos_zero).sum())

print(f"Rows before filter: {n_before:,}")
print(f"Excluding position NaN (no GSC data days 1-15): {n_pos_nan:,}")
print(f"Excluding position == 0 (the 'no data' placeholder): {n_pos_zero:,}")
print(f"Excluding undefined labels (no GSC data in one half): {n_lab_nan_extra:,}")
print("Choice: undefined-label pages are dropped, not labelled declining or not declining.")

df = df[~pos_nan & ~pos_zero & ~lab_nan].copy()
df["is_declining_label"] = df["is_declining_label"].astype(int)
df = df.reset_index(drop=True)
# Canonical row order (content_id is unique) so folds, tied queues and the
# forest's bootstrap reproduce in any environment (same as Week 5).
df = df.sort_values("content_id", kind="stable").reset_index(drop=True)
n_dropped = n_before - len(df)
assert n_dropped == n_pos_nan + n_pos_zero + n_lab_nan_extra

df["ctr"] = np.where(df["gsc_impressions_avg"] > 0,
                     df["gsc_clicks_avg"] / df["gsc_impressions_avg"], 0.0)

# Reference rankings (no training step - they must not move between splits)
flag = ((df["gsc_impressions_avg"] >= IMPRESSIONS_THRESHOLD) &
        (df["position_avg"] >= POSITION_THRESHOLD)).to_numpy()
df["rule_full"] = df["gsc_impressions_avg"] + 1e9 * flag
df["impressions_only"] = df["gsc_impressions_avg"]

# Week-4/Week-5 universe: assert the exact numbers I already published.
assert len(df) == 140_599, f"expected 140,599 pages, got {len(df):,}"
assert int(flag.sum()) == 1_074, f"expected 1,074 flagged pages, got {int(flag.sum()):,}"
base_rate = df["is_declining_label"].mean()
assert abs(base_rate - 0.465) < 0.0005, f"expected base rate 0.465, got {base_rate:.3f}"

print(f"\nModeling frame: {len(df):,} pages, {df['client_hash_id'].nunique()} clients")
print(f"Flagged by the rule: {int(flag.sum()):,} ({flag.mean():.1%})")
print(f"Base rate (share declining): {base_rate:.3f}")
print("Universe matches Week 4 / Week 5 exactly.")

Rows before filter: 176,738
Excluding position NaN (no GSC data days 1-15): 24,757
Excluding position == 0 (the 'no data' placeholder): 1,306
Excluding undefined labels (no GSC data in one half): 10,076
Choice: undefined-label pages are dropped, not labelled declining or not declining.

Modeling frame: 140,599 pages, 43 clients
Flagged by the rule: 1,074 (0.8%)
Base rate (share declining): 0.465
Universe matches Week 4 / Week 5 exactly.


In [6]:
# ---- The two splits ---------------------------------------------------------
def make_grouped_folds(frame, n_splits=N_FOLDS, seed=RANDOM_STATE):
    # Client-grouped: every client lands in exactly one test fold (Week 5's split).
    sizes = frame.groupby("client_hash_id").size()
    order = list(sizes.index)
    np.random.default_rng(seed).shuffle(order)      # shuffle so ties are not alphabetical
    fold_of_client, load = {}, [0] * n_splits
    for client in order:
        i = int(np.argmin(load))                    # always fill the lightest fold
        fold_of_client[client] = i
        load[i] += int(sizes[client])
    return frame["client_hash_id"].map(fold_of_client).astype(int)

def make_row_folds(frame, n_splits=N_FOLDS, seed=RANDOM_STATE):
    # Row-random: the naive default - rows shuffled, folds cut by row.
    idx = np.random.default_rng(seed).permutation(len(frame))
    fold = np.empty(len(frame), dtype=int)
    for i, part in enumerate(np.array_split(idx, n_splits)):
        fold[part] = i
    return pd.Series(fold, index=frame.index).astype(int)

df["fold_row"] = make_row_folds(df)
df["fold_grouped"] = make_grouped_folds(df)

assert df.groupby("client_hash_id")["fold_grouped"].nunique().eq(1).all(), "a client spans two grouped folds"
for f in range(N_FOLDS):
    assert df.loc[df["fold_grouped"] == f, "is_declining_label"].nunique() == 2, f"fold {f} has one class only"

# Diagnostics: what each split is actually testing
rows = []
for name in ["fold_row", "fold_grouped"]:
    for f in range(N_FOLDS):
        test = df[name] == f
        train = ~test
        test_clients = set(df.loc[test, "client_hash_id"])
        train_clients = set(df.loc[train, "client_hash_id"])
        both = test_clients & train_clients
        rows.append({
            "split": name, "fold": f,
            "test_pages": int(test.sum()),
            "test_clients": len(test_clients),
            "clients_in_train_AND_test": len(both),
            "%_test_rows_client_seen_in_train":
                round(100 * df.loc[test, "client_hash_id"].isin(train_clients).mean(), 1),
        })
display(pd.DataFrame(rows))

assert df["fold_row"].nunique() == N_FOLDS and df["fold_grouped"].nunique() == N_FOLDS
print("\nrow-random: test pages whose client was also in training (the memorization door)")
print("grouped:    0 clients in both sides, by construction")

,split,fold,test_pages,test_clients,clients_in_train_AND_test,%_test_rows_client_seen_in_train
0,fold_row,0,28120,41,41,100.0
1,fold_row,1,28120,41,41,100.0
2,fold_row,2,28120,41,41,100.0
3,fold_row,3,28120,42,42,100.0
4,fold_row,4,28119,41,41,100.0
5,fold_grouped,0,32790,8,0,0.0
6,fold_grouped,1,25826,9,0,0.0
7,fold_grouped,2,26662,7,0,0.0
8,fold_grouped,3,32230,12,0,0.0
9,fold_grouped,4,23091,7,0,0.0



row-random: test pages whose client was also in training (the memorization door)
grouped:    0 clients in both sides, by construction


In [7]:
# ---- Metric helpers + the one OOF runner ------------------------------------
def rule_score(frame):
    # My Week-4 rule: flagged pages ranked by impressions, everything else 0.
    flagged = (frame["gsc_impressions_avg"] >= IMPRESSIONS_THRESHOLD) & \
              (frame["position_avg"] >= POSITION_THRESHOLD)
    return np.where(flagged, frame["gsc_impressions_avg"], 0.0)

def precision_at_k(labels, scores, k):
    order = np.argsort(-np.asarray(scores), kind="stable")[:k]
    return float(np.asarray(labels)[order].mean())

def metric_bundle(labels, scores):
    return {
        "precision_at_10": precision_at_k(labels, scores, 10),
        "precision_at_20": precision_at_k(labels, scores, 20),
        "precision_at_50": precision_at_k(labels, scores, 50),
        "roc_auc": roc_auc_score(labels, scores),
        "avg_precision": average_precision_score(labels, scores),
    }

def build_models():
    return {
        "logistic_regression": Pipeline([
            ("log", ColumnTransformer([("log", FunctionTransformer(np.log1p), [0, 1])],
                                      remainder="passthrough")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(class_weight="balanced", max_iter=1000,
                                         random_state=RANDOM_STATE)),
        ]),
        "random_forest": RandomForestClassifier(
            n_estimators=200, max_depth=10, min_samples_leaf=25,
            class_weight="balanced_subsample", n_jobs=-1,
            random_state=RANDOM_STATE),
    }

def run_split(fold_col, feature_cols, prefix):
    # Out-of-fold scoring under one split. Every page is scored exactly once.
    X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    y = df["is_declining_label"].to_numpy()
    scores = {"rule": rule_score(df),
              "rule_full": df["rule_full"].to_numpy(),
              "impressions_only": df["impressions_only"].to_numpy(),
              "logistic_regression": np.zeros(len(df)),
              "random_forest": np.zeros(len(df))}
    fold_rows, fold_models = [], []

    for f in range(N_FOLDS):
        test = (df[fold_col] == f).to_numpy()
        train = ~test
        models = build_models()
        for name in ("logistic_regression", "random_forest"):
            models[name].fit(X.iloc[train], y[train])
            scores[name][test] = models[name].predict_proba(X.iloc[test])[:, 1]
        fold_models.append(models)

        row = {"fold": f, "pages": int(test.sum()), "base_rate": float(y[test].mean())}
        for name in scores:
            for key, value in metric_bundle(y[test], scores[name][test]).items():
                row[f"{name}__{key}"] = value
        fold_rows.append(row)

    pooled = {name: metric_bundle(y, s) for name, s in scores.items()}
    for name, values in scores.items():
        df[f"{prefix}__{name}"] = values
    return pd.DataFrame(fold_rows), pooled, fold_models

print("Helpers defined. run_split(fold_col, feature_cols, prefix) scores every page out-of-fold once.")

Helpers defined. run_split(fold_col, feature_cols, prefix) scores every page out-of-fold once.


In [8]:
# ---- BEFORE (row-random) and AFTER (client-grouped), same everything else ---
before_folds, before_pooled, before_models = run_split("fold_row", BASE_FEATURES, "before")
after_folds,  after_pooled,  after_models  = run_split("fold_grouped", BASE_FEATURES, "after")

def table(pooled):
    return pd.DataFrame(pooled).T.loc[:, METRIC_ORDER].round(3)

print("=" * 78)
print("BEFORE - row-random 5-fold (the naive default)")
display(table(before_pooled))
print(f"base rate: {base_rate:.3f}   pages: {len(df):,}   (positive class = declining)")

print("=" * 78)
print("AFTER - client-grouped 5-fold (honest: test clients never seen in training)")
display(table(after_pooled))
print(f"base rate: {base_rate:.3f}   pages: {len(df):,}   clients: {df['client_hash_id'].nunique()}")

print("=" * 78)
print("BEFORE - AFTER  (positive = the row-random split flattered the model)")
delta = (pd.DataFrame(before_pooled).T - pd.DataFrame(after_pooled).T).loc[:, METRIC_ORDER]
display(delta.round(3))

# The harness sanity check: rows with no training step must not move at all.
for m in ("rule", "rule_full", "impressions_only"):
    assert delta.loc[m].abs().max() < 1e-12, f"{m} moved between splits - harness broken"
print("\nCheck: rule / rule_full / impressions_only identical under both splits (no training step).")

BEFORE - row-random 5-fold (the naive default)


,precision_at_10,precision_at_20,precision_at_50,roc_auc,avg_precision
rule,1.0,0.95,0.88,0.505,0.469
rule_full,1.0,0.95,0.88,0.609,0.537
impressions_only,0.7,0.70,0.62,0.609,0.534
logistic_regression,0.8,0.75,0.82,0.634,0.575
random_forest,0.7,0.60,0.80,0.672,0.609


base rate: 0.465   pages: 140,599   (positive class = declining)
AFTER - client-grouped 5-fold (honest: test clients never seen in training)


,precision_at_10,precision_at_20,precision_at_50,roc_auc,avg_precision
rule,1.0,0.95,0.88,0.505,0.469
rule_full,1.0,0.95,0.88,0.609,0.537
impressions_only,0.7,0.70,0.62,0.609,0.534
logistic_regression,0.7,0.70,0.78,0.630,0.571
random_forest,0.5,0.55,0.66,0.654,0.583


base rate: 0.465   pages: 140,599   clients: 43
BEFORE - AFTER  (positive = the row-random split flattered the model)


,precision_at_10,precision_at_20,precision_at_50,roc_auc,avg_precision
rule,0.0,0.00,0.00,0.000,0.000
rule_full,0.0,0.00,0.00,0.000,0.000
impressions_only,0.0,0.00,0.00,0.000,0.000
logistic_regression,0.1,0.05,0.04,0.004,0.004
random_forest,0.2,0.05,0.14,0.017,0.026



Check: rule / rule_full / impressions_only identical under both splits (no training step).


In [9]:
# ---- Per-fold tables + spread ----------------------------------------------
print("BEFORE - row-random, per fold:")
display(before_folds.round(3))
print("AFTER - client-grouped, per fold (each fold = unseen clients):")
display(after_folds.round(3))

print("Spread across folds (mean +/- std), the two learned models:")
for name in ("logistic_regression", "random_forest"):
    b50 = before_folds[f"{name}__precision_at_50"]
    a50 = after_folds[f"{name}__precision_at_50"]
    bau = before_folds[f"{name}__roc_auc"]
    aau = after_folds[f"{name}__roc_auc"]
    print(f"  {name:<22} P@50  before {b50.mean():.3f} +/- {b50.std():.3f}   "
          f"after {a50.mean():.3f} +/- {a50.std():.3f}")
    print(f"  {'':<22} AUC   before {bau.mean():.3f} +/- {bau.std():.3f}   "
          f"after {aau.mean():.3f} +/- {aau.std():.3f}")

BEFORE - row-random, per fold:


,fold,pages,base_rate,rule__precision_at_10,rule__precision_at_20,rule__precision_at_50,rule__roc_auc,rule__avg_precision,rule_full__precision_at_10,rule_full__precision_at_20,...,logistic_regression__precision_at_10,logistic_regression__precision_at_20,logistic_regression__precision_at_50,logistic_regression__roc_auc,logistic_regression__avg_precision,random_forest__precision_at_10,random_forest__precision_at_20,random_forest__precision_at_50,random_forest__roc_auc,random_forest__avg_precision
0,0,28120,0.463,0.8,0.80,0.82,0.504,0.467,0.8,0.80,...,0.7,0.75,0.74,0.631,0.569,0.7,0.80,0.82,0.672,0.608
1,1,28120,0.467,0.9,0.80,0.82,0.505,0.472,0.9,0.80,...,0.9,0.80,0.78,0.633,0.576,0.8,0.80,0.92,0.670,0.610
2,2,28120,0.463,0.8,0.85,0.78,0.504,0.467,0.8,0.85,...,0.7,0.80,0.80,0.634,0.575,0.8,0.80,0.86,0.671,0.607
3,3,28120,0.461,1.0,1.00,0.86,0.505,0.467,1.0,1.00,...,0.7,0.80,0.84,0.636,0.574,0.8,0.75,0.80,0.676,0.608
4,4,28119,0.470,0.9,0.85,0.86,0.505,0.475,0.9,0.85,...,1.0,0.85,0.82,0.634,0.581,0.9,0.85,0.86,0.669,0.610


AFTER - client-grouped, per fold (each fold = unseen clients):


,fold,pages,base_rate,rule__precision_at_10,rule__precision_at_20,rule__precision_at_50,rule__roc_auc,rule__avg_precision,rule_full__precision_at_10,rule_full__precision_at_20,...,logistic_regression__precision_at_10,logistic_regression__precision_at_20,logistic_regression__precision_at_50,logistic_regression__roc_auc,logistic_regression__avg_precision,random_forest__precision_at_10,random_forest__precision_at_20,random_forest__precision_at_50,random_forest__roc_auc,random_forest__avg_precision
0,0,32790,0.487,0.8,0.80,0.70,0.502,0.489,0.8,0.80,...,0.7,0.80,0.80,0.653,0.605,0.7,0.70,0.74,0.657,0.605
1,1,25826,0.439,0.4,0.45,0.46,0.500,0.440,0.4,0.45,...,0.5,0.50,0.46,0.589,0.493,0.5,0.65,0.60,0.623,0.522
2,2,26662,0.547,0.6,0.50,0.56,0.500,0.547,0.6,0.50,...,0.9,0.85,0.72,0.606,0.627,0.6,0.70,0.68,0.640,0.644
3,3,32230,0.353,0.8,0.70,0.60,0.501,0.353,0.8,0.70,...,0.5,0.60,0.58,0.595,0.422,0.7,0.65,0.68,0.629,0.446
4,4,23091,0.524,1.0,0.95,0.86,0.521,0.542,1.0,0.95,...,0.9,0.85,0.80,0.659,0.655,1.0,0.90,0.88,0.699,0.681


Spread across folds (mean +/- std), the two learned models:
  logistic_regression    P@50  before 0.796 +/- 0.038   after 0.672 +/- 0.149
                         AUC   before 0.634 +/- 0.001   after 0.620 +/- 0.033
  random_forest          P@50  before 0.852 +/- 0.046   after 0.716 +/- 0.104
                         AUC   before 0.672 +/- 0.003   after 0.650 +/- 0.031


In [10]:
# ---- Cross-check: does AFTER reproduce what I published in Weeks 4 and 5? ---
ref_path = Path("work/outputs/baseline_metrics_latest.json")
if not ref_path.exists():
    try:
        import urllib.request
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/musab855/"
            "flyrank-ml-internship/main/work/outputs/baseline_metrics_latest.json",
            ref_path)
        print("Fetched baseline json from GitHub (fresh runtime).")
    except Exception as exc:
        print("Could not fetch baseline json:", exc)

if ref_path.exists():
    ref = json.loads(ref_path.read_text())
    checks = [("precision_at_10", after_pooled["rule"]["precision_at_10"]),
              ("precision_at_20", after_pooled["rule"]["precision_at_20"]),
              ("precision_at_50", after_pooled["rule"]["precision_at_50"]),
              ("base_rate", base_rate)]
    print("Week-4 baseline json  vs  this notebook's AFTER rule row")
    for key, mine in checks:
        match = abs(float(ref[key]) - float(mine)) < 5e-4
        print(f"  {key:<16} json {float(ref[key]):.3f}   notebook {float(mine):.3f}   "
              f"{'MATCH' if match else 'DIFFERS'}")
    print(f"  {'total_pages':<16} json {int(ref['total_pages']):,}   notebook {len(df):,}")
    print(f"  {'flagged_pages':<16} json {int(ref['flagged_pages']):,}   notebook {int(flag.sum()):,}")
else:
    print("work/outputs/baseline_metrics_latest.json not found - skipping the json cross-check.")

# Week 5's published pooled table (w05_model.ipynb, section 3) vs my AFTER run.
W05_PUBLISHED = {
    "rule":               {"precision_at_10": 1.000, "precision_at_20": 0.950, "precision_at_50": 0.880, "roc_auc": 0.505, "avg_precision": 0.469},
    "rule_full":          {"precision_at_10": 1.000, "precision_at_20": 0.950, "precision_at_50": 0.880, "roc_auc": 0.609, "avg_precision": 0.537},
    "impressions_only":   {"precision_at_10": 0.700, "precision_at_20": 0.700, "precision_at_50": 0.620, "roc_auc": 0.609, "avg_precision": 0.534},
    "logistic_regression": {"precision_at_10": 0.700, "precision_at_20": 0.700, "precision_at_50": 0.780, "roc_auc": 0.630, "avg_precision": 0.571},
    "random_forest":      {"precision_at_10": 0.500, "precision_at_20": 0.550, "precision_at_50": 0.660, "roc_auc": 0.654, "avg_precision": 0.583},
}
print("\nWeek 5's published pooled table  vs  this notebook's AFTER (client-grouped) run:")
n_match = n_total = 0
for method, want in W05_PUBLISHED.items():
    for key, want_val in want.items():
        got = after_pooled[method][key]
        n_total += 1
        ok = abs(got - want_val) < 0.005
        n_match += int(ok)
        print(f"  {method:<21} {key:<16} w05 {want_val:.3f}   here {got:.3f}   "
              f"{'MATCH' if ok else 'DIFFERS'}")
print(f"\n{n_match} of {n_total} published numbers reproduced (tolerance 0.005).")

Week-4 baseline json  vs  this notebook's AFTER rule row
  precision_at_10  json 1.000   notebook 1.000   MATCH
  precision_at_20  json 0.950   notebook 0.950   MATCH
  precision_at_50  json 0.880   notebook 0.880   MATCH
  base_rate        json 0.465   notebook 0.465   MATCH
  total_pages      json 140,599   notebook 140,599
  flagged_pages    json 1,074   notebook 1,074

Week 5's published pooled table  vs  this notebook's AFTER (client-grouped) run:
  rule                  precision_at_10  w05 1.000   here 1.000   MATCH
  rule                  precision_at_20  w05 0.950   here 0.950   MATCH
  rule                  precision_at_50  w05 0.880   here 0.880   MATCH
  rule                  roc_auc          w05 0.505   here 0.505   MATCH
  rule                  avg_precision    w05 0.469   here 0.469   MATCH
  rule_full             precision_at_10  w05 1.000   here 1.000   MATCH
  rule_full             precision_at_20  w05 0.950   here 0.950   MATCH
  rule_full             precision_at_50

### Reading the before/after

**The gap, in one view** (pooled out-of-fold, all 140,599 pages, base rate **0.465** printed under both tables above):

| method | metric | BEFORE row-random | AFTER client-grouped | gap (before − after) |
|---|---|---:|---:|---:|
| logistic regression | P@50 | 0.820 | 0.780 | +0.040 |
| logistic regression | ROC-AUC | 0.634 | 0.630 | +0.004 |
| random forest | P@10 | 0.700 | 0.500 | **+0.200** |
| random forest | P@50 | 0.800 | 0.660 | **+0.140** |
| random forest | ROC-AUC | 0.672 | 0.654 | +0.017 |
| random forest | avg precision | 0.609 | 0.583 | +0.026 |
| rule / rule_full / impressions_only | all metrics | identical | identical | 0.000 |

gap = before − after on unrounded values (what the delta table prints). The forest ROC-AUC row reads +0.017 while 0.672 − 0.654 = 0.018 — subtracting two already-rounded values, a rounding artifact, not a different measurement. Every other row matches its subtraction exactly.

- **The naive split flattered the forest most, and worst where it matters most — the top of the queue:** P@10 +0.200, P@50 +0.140 of precision that came from training on clients the model was also tested on. Every learned metric is ≥ under BEFORE; not one improved under AFTER.
- **The diagnostics cell shows the mechanism:** under the row split, **100.0%** of test rows had their client also in training (41–42 of 43 clients on both sides of every fold); under the grouped split, **0**. The gap is the memorization that was available, measured.
- **AFTER reproduces what I already published:** **25 of 25** numbers from Week 5's pooled table MATCH (tolerance 0.005), and the Week-4 baseline json cross-check is 4/4 MATCH on 140,599 pages / 1,074 flagged / base rate 0.465. The honest number I reported in Week 5 survives being re-derived from scratch here.

**What the gap does not mean.** It does not mean the row-random arithmetic was broken — it answered *"does it work on pages whose client the model has already seen?"*, a question nobody deploys against. And it does not show that the grouped number will hold on another month or another portfolio; that remains unmeasured here.

**Why the AFTER table is the one that ships.** The grouped number is the only one that answers the deployment question — *does it work on a client it never saw?* The row-random number answers an easier question nobody asked.

### Failure examples — real rows the honest model got wrong

*Error examples from the AFTER split: the forest's best-scoring pages that did not decline.*

In [11]:
# ---- Concrete wrong rows, from the client-grouped out-of-fold scores --------
top50 = df.sort_values("after__random_forest", ascending=False, kind="stable").head(50)
n_wrong = int((top50["is_declining_label"] == 0).sum())
false_pos = top50[top50["is_declining_label"] == 0].head(3)

def why_hard(row):
    bits = []
    if row["ctr"] < 0.001:
        bits.append(f"near-zero CTR ({row['ctr']:.4f}) on {row['gsc_impressions_avg']:,.0f} "
                    f"impressions/day - may be bot or indexing noise rather than a real audience")
    if row["position_avg"] >= 30:
        bits.append(f"position {row['position_avg']:.0f} is far off page one, where impression "
                    f"counts swing hard day to day")
    if not bits:
        bits.append("its days 1-15 numbers looked like the pages that did decline; "
                    "days 16-31 simply held steady")
    bits.append("and the label only asks whether impressions fell at all - a page can hold "
                "steady and still be a bad page, which this label cannot see")
    return "; ".join(bits)

print(f"The forest's top-50 (pooled, client-grouped OOF) contains {n_wrong} pages that did NOT decline.")
print(f"Base rate on the frame: {base_rate:.3f} - so a random 50 would carry about "
      f"{50 * base_rate:.0f} declining pages.\n")

for _, row in false_pos.iterrows():
    print(f"  {row['content_id']}")
    print(f"    impressions/day {row['gsc_impressions_avg']:,.0f} | clicks/day "
          f"{row['gsc_clicks_avg']:,.1f} | position {row['position_avg']:.1f} | "
          f"forest score {row['after__random_forest']:.3f} | declining? no")
    print(f"    why it is hard: {why_hard(row)}\n")

# How concentrated is that top-50?
print(f"clients represented in the forest's top-50: {top50['client_hash_id'].nunique()}")
print(f"largest single client's share of it: {top50['client_hash_id'].value_counts(normalize=True).iloc[0]:.0%}")

# Same rows under the BEFORE split - does the naive split change who gets queued?
top50_before = df.sort_values("before__random_forest", ascending=False, kind="stable").head(50)
overlap = len(set(top50["content_id"]) & set(top50_before["content_id"]))
print(f"\ntop-50 overlap between BEFORE and AFTER splits: {overlap} of 50")
print(f"BEFORE top-50 declining: {int(top50_before['is_declining_label'].sum())} | "
      f"AFTER top-50 declining: {int(top50['is_declining_label'].sum())}")

The forest's top-50 (pooled, client-grouped OOF) contains 17 pages that did NOT decline.
Base rate on the frame: 0.465 - so a random 50 would carry about 23 declining pages.

  content_a11bd5663919f057
    impressions/day 2,065 | clicks/day 1.7 | position 40.8 | forest score 0.937 | declining? no
    why it is hard: near-zero CTR (0.0008) on 2,065 impressions/day - may be bot or indexing noise rather than a real audience; position 41 is far off page one, where impression counts swing hard day to day; and the label only asks whether impressions fell at all - a page can hold steady and still be a bad page, which this label cannot see

  content_c7da032c307b766a
    impressions/day 860 | clicks/day 1.3 | position 35.9 | forest score 0.928 | declining? no
    why it is hard: position 36 is far off page one, where impression counts swing hard day to day; and the label only asks whether impressions fell at all - a page can hold steady and still be a bad page, which this label cannot see

  c

- **17 of the forest's pooled top-50 did not decline** (base rate 0.465 — a random queue of 50 would carry about 23). The three printed rows share one shape: positions 35.9–40.8, 860–2,065 impressions/day, but only 0.5–1.7 clicks/day (CTR 0.0003–0.0008), scored 0.913–0.937. They are hard because the label can only ask *"did impressions fall?"* — a page can hold steady and still be worthless, and at position ~40 the impression count itself swings day to day.
- **The split changes who gets queued, not just the metrics.** The BEFORE and AFTER top-50s share only **5 of 50** pages, and BEFORE's top-50 held **40** declining pages against AFTER's **33**. So the naive split did not merely inflate a score — it produced a *different queue*, one assembled with client overlap. That is the most concrete argument in this notebook for the grouped split: two different sets of pages would reach a reviewer.
- **Both queues are client-concentrated:** 4 clients in the AFTER top-50, largest single client 68% — a property of the data (the top client owns most of the flagged pages), as noted in Week 5, not a property of the model.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Three tests from the leakage taxonomy, all run against the frame this notebook actually uses:

1. **Label-derived** — assert that no label, label-sibling, trend, or ID column sits in the feature list; print what is in versus what is banned.
2. **Future / overlapping window — the deliberate confession.** Hand the model `gsc_impressions_16to31` (the label's own window, kept in the cache only for this test) and watch the harness react *loudly*; then remove it and keep the honest number. Per the skill: if adding a leaky feature did NOT make the score jump, the test harness itself would be broken.
3. **Decision-derived** — the rule's thresholds produce baseline rows only; no product flag or existing-system score enters the feature matrix.

Every table below also prints the base rate, and the timeline is printed rather than assumed.

In [12]:
# ---- Test 1: label-derived columns + the timeline ---------------------------
BANNED_FEATURES = {
    "content_id", "client_hash_id",             # IDs: grouping only, never features
    "is_declining_label", "ctr",                # the label itself, and a same-window ratio
    "trend_pct", "trend_direction",             # the label's ancestors (Week 3's trap)
    "gsc_impressions_16to31",                   # the label window itself
    "rule", "rule_full", "impressions_only",    # baseline scores, not inputs
    "fold_row", "fold_grouped", "ctr",          # bookkeeping columns
}

violations = set(BASE_FEATURES) & BANNED_FEATURES
assert not violations, f"banned columns in the feature list: {violations}"
assert set(BASE_FEATURES).isdisjoint(set(AUDIT_ONLY)), "audit column leaked into BASE_FEATURES"
print("features used by EVERY model in this notebook:", BASE_FEATURES)
print("banned from ever being a feature:", sorted(BANNED_FEATURES))
print("violations:", violations or "none")

timeline = pd.DataFrame([
    {"stage": "features",  "window": "2026-03-01 .. 2026-03-15",
     "knowable at prediction moment?": "yes - strictly before it"},
    {"stage": "prediction", "window": "2026-03-16",
     "knowable at prediction moment?": "this IS the moment"},
    {"stage": "label",     "window": "2026-03-16 .. 2026-03-31",
     "knowable at prediction moment?": "no - strictly after (only the audit column touches it)"},
])
print("\nTimeline (drawn, not assumed):")
display(timeline)

# Kin-of-target: impressions is the label's day 1-15 ancestor. Expected to lead -
# disclosed in Week 5, section 1. It is measured BEFORE the label window, so it is
# legal; it is just close kin, which is why it dominates every importance measure.
kin = (df[BASE_FEATURES + ["is_declining_label"]]
       .corr()["is_declining_label"]
       .drop("is_declining_label")
       .sort_values(ascending=False))
print("\nCorrelation of each feature with the label (kin-of-target check):")
display(kin.round(3))
print("impressions leads - expected: the label IS an impressions drop, and the feature is its "
      "day 1-15 ancestor. Legal (measured first), close kin, disclosed.")

features used by EVERY model in this notebook: ['gsc_impressions_avg', 'gsc_clicks_avg', 'position_avg']
banned from ever being a feature: ['client_hash_id', 'content_id', 'ctr', 'fold_grouped', 'fold_row', 'gsc_impressions_16to31', 'impressions_only', 'is_declining_label', 'rule', 'rule_full', 'trend_direction', 'trend_pct']
violations: none

Timeline (drawn, not assumed):


,stage,window,knowable at prediction moment?
0,features,2026-03-01 .. 2026-03-15,yes - strictly before it
1,prediction,2026-03-16,this IS the moment
2,label,2026-03-16 .. 2026-03-31,no - strictly after (only the audit column tou...



Correlation of each feature with the label (kin-of-target check):


,is_declining_label
gsc_impressions_avg,0.056
gsc_clicks_avg,-0.019
position_avg,-0.045


impressions leads - expected: the label IS an impressions drop, and the feature is its day 1-15 ancestor. Legal (measured first), close kin, disclosed.


In [13]:
# ---- Test 2: the deliberate confession (future-window leakage) --------------
# The skill's harness check: ADD the leaky feature on purpose. The label is
# literally  impressions_16to31 < impressions_1to15  - so giving the model both
# columns is giving it the answer sheet.
print("Is the audit column really the label? Recomputing the label from it:")
recomputed = (df["gsc_impressions_16to31"] < df["gsc_impressions_avg"]).astype(int)
agree = float((recomputed == df["is_declining_label"]).mean())
print(f"  agreement: {agree:.4%}  (the label is DEFINED as this comparison)")
assert agree > 0.9999, "audit column does not reconstruct the label - check the SQL"

leaky_features = BASE_FEATURES + AUDIT_ONLY
leaky_folds, leaky_pooled, _ = run_split("fold_grouped", leaky_features, "leaky")
# run_split wrote leaky scores into df - that is fine, they are never used again;
# BASE_FEATURES is untouched.

comp = pd.concat(
    [table(after_pooled).loc[["logistic_regression", "random_forest"], METRIC_ORDER],
     table(leaky_pooled).loc[["logistic_regression", "random_forest"], METRIC_ORDER]],
    axis=1, keys=["honest (3 features)", "+ label-window column (LEAKY)"])
print("\nPooled out-of-fold, client-grouped, honest vs deliberately leaky:")
display(comp)

gain = (pd.DataFrame(leaky_pooled).T - pd.DataFrame(after_pooled).T).loc[
    ["logistic_regression", "random_forest"], ["roc_auc", "avg_precision"]]
print("\nLeaky - honest (this is what a leak LOOKS like):")
display(gain.round(3))

assert set(BASE_FEATURES) == set(["gsc_impressions_avg", "gsc_clicks_avg", "position_avg"])
print("\nRemoval: the audit column is NOT in BASE_FEATURES. The honest numbers in section 2 "
      "stand; the leaky run exists only to prove this harness would catch one.")

Is the audit column really the label? Recomputing the label from it:
  agreement: 100.0000%  (the label is DEFINED as this comparison)

Pooled out-of-fold, client-grouped, honest vs deliberately leaky:


honest (3 features)                                  \
                        precision_at_10 precision_at_20 precision_at_50   
logistic_regression                 0.7            0.70            0.78   
random_forest                       0.5            0.55            0.66   

                                          + label-window column (LEAKY)  \
                    roc_auc avg_precision               precision_at_10   
logistic_regression   0.630         0.571                           1.0   
random_forest         0.654         0.583                           1.0   

                                                                           
                    precision_at_20 precision_at_50 roc_auc avg_precision  
logistic_regression             1.0             1.0   0.698         0.711  
random_forest                   1.0             1.0   0.997         0.996


Leaky - honest (this is what a leak LOOKS like):


,roc_auc,avg_precision
logistic_regression,0.068,0.140
random_forest,0.342,0.413



Removal: the audit column is NOT in BASE_FEATURES. The honest numbers in section 2 stand; the leaky run exists only to prove this harness would catch one.


- **The audit column *is* the label:** it reconstructs it **100.0000%** of the time, by construction (`label = impressions_16to31 < impressions_1to15`). Feeding it to the model is handing over the answer sheet.
- **The harness reacts loudly — the confession the skill asks for:** the forest's ROC-AUC goes **0.654 → 0.997** (+0.342), average precision 0.583 → 0.996, and **P@50 → 1.000** — a perfect queue. The linear model cannot express "x < y" as one clean boundary, so it only reaches AUC 0.698 — yet its P@50 also hits **1.000**: the top of the queue is fully poisoned either way.
- **This is the skill's harness check passing:** *if adding a leaky feature did NOT make the score jump, the test harness itself would be broken.* Here it jumps; the column is then removed, `BASE_FEATURES` is asserted back to its three honest members, and every number in section 2 stands without it.

In [14]:
# ---- Test 3: decision-derived features + train-without-the-kin --------------
# (a) No product flag / existing-system score in the inputs: the rule's thresholds
#     only ever produce the baseline ROWS (rule / rule_full), never a feature.
rule_derived = [c for c in df.columns if "rule" in c or "flag" in c or "threshold" in c]
print("rule/flag-derived columns in the frame:", rule_derived)
print("of those, in BASE_FEATURES:", set(rule_derived) & set(BASE_FEATURES) or "none - baseline only")

# (b) Train-without-the-suspect test: drop the kin-of-target and see what is left.
no_imp_folds, no_imp_pooled, _ = run_split(
    "fold_grouped", ["gsc_clicks_avg", "position_avg"], "noimp")

compare = pd.concat(
    [table(after_pooled).loc[["logistic_regression", "random_forest"], METRIC_ORDER],
     table(no_imp_pooled).loc[["logistic_regression", "random_forest"], METRIC_ORDER]],
    axis=1, keys=["full (3 features)", "without impressions"]).round(3)
print("\nTrain WITH vs WITHOUT the kin-of-target feature (client-grouped):")
display(compare)
print("impressions carries most of the signal - disclosed, expected, and measured before the "
      "label window. Without it: logistic lands at 0.494 (at/below the 0.500 no-skill line), "
      "forest at 0.591. Oddly the forest's P@50 rises without it (0.740 vs 0.660) - a single-K "
      "queue stat can move against the whole-list evidence; the reading cell below says so.")

# (c) What the honest forest leans on (impurity importance, mean across grouped folds)
imp = pd.DataFrame([dict(zip(BASE_FEATURES, m["random_forest"].feature_importances_))
                    for m in after_models]).mean()
print("\nForest impurity importance (mean over the 5 grouped folds, shares of 1):")
display(imp.round(3))
print("Sanity: no feature towers toward 1.0, and the honest AUC is 0.654 - not 0.99. "
      "'Too good' would be a bug, not a win.")

rule/flag-derived columns in the frame: ['rule_full', 'before__rule', 'before__rule_full', 'after__rule', 'after__rule_full', 'leaky__rule', 'leaky__rule_full']
of those, in BASE_FEATURES: none - baseline only

Train WITH vs WITHOUT the kin-of-target feature (client-grouped):


full (3 features)                                          \
                      precision_at_10 precision_at_20 precision_at_50 roc_auc   
logistic_regression               0.7            0.70            0.78   0.630   
random_forest                     0.5            0.55            0.66   0.654   

                                  without impressions                  \
                    avg_precision     precision_at_10 precision_at_20   
logistic_regression         0.571                 0.5             0.4   
random_forest               0.583                 0.6             0.7   

                                                           
                    precision_at_50 roc_auc avg_precision  
logistic_regression            0.36   0.494         0.459  
random_forest                  0.74   0.591         0.536

impressions carries most of the signal - disclosed, expected, and measured before the label window. Without it: logistic lands at 0.494 (at/below the 0.500 no-skill line), forest at 0.591. Oddly the forest's P@50 rises without it (0.740 vs 0.660) - a single-K queue stat can move against the whole-list evidence; the reading cell below says so.

Forest impurity importance (mean over the 5 grouped folds, shares of 1):


,0
gsc_impressions_avg,0.749
gsc_clicks_avg,0.092
position_avg,0.159


Sanity: no feature towers toward 1.0, and the honest AUC is 0.654 - not 0.99. 'Too good' would be a bug, not a win.


- **Decision-derived:** the rule-derived columns that exist (`rule_full` plus the score columns written by the runners) — **none of them is in `BASE_FEATURES`**. The thresholds only ever produce baseline *rows*, never model *inputs*.
- **Train-without-the-kin:** dropping impressions leaves logistic regression at **ROC-AUC 0.494** — at/below the 0.500 no-skill line — and the forest at **0.591** (from 0.654). So impressions carries the bulk of the signal; it is legal (measured days 1–15, strictly before the label window) and has been disclosed as close kin of the target since Week 5, section 1.
- **One honest oddity, reported not hidden:** without impressions the forest's **P@50 reads 0.740 — higher than the full model's 0.660** — while every whole-list metric (AUC 0.654 → 0.591, AP 0.583 → 0.536) falls. Precision at a single K is a noisy, queue-composition statistic; it can move against the whole-list evidence. This is exactly why I report P@10/20/50 *next to* ROC-AUC and average precision rather than choosing whichever looks best.
- **What the honest forest leans on:** impurity importance, mean over the 5 grouped folds — impressions **0.749**, position **0.159**, clicks **0.092**. No feature towers toward 1.0, and the honest AUC stays at 0.654 rather than 0.99: "too good" would be a bug, not a win.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Three sentences from my own Week-5 notebook, then a scan of *this* notebook's markdown for phrasings that claim more than any observational design can carry.

In [15]:
# ---- (a) My boldest sentences -> what the evidence carries ------------------
CLAIMS = [
    ("Week 5, section 3, final line",
     "the forest is a modest, genuine ranking improvement over what impressions alone can do",
     "one month of data, one portfolio, client-grouped OOF, AUC 0.654 vs 0.609 - a measured gap of +0.045",
     "On this March 2026 frame, under client-grouped out-of-fold scoring, the forest was measured at "
     "ROC-AUC 0.654 vs 0.609 for the impressions-only rank - an observed +0.045. Directional, "
     "decision-support: it reorders a review queue; it does not predict Google."),

    ("Week 5, section 3",
     "Did the model beat the baseline? ... on the whole list against a fair version of the rule: yes.",
     "true only for whole-list metrics on this frame; at P@50 the rule still leads 0.880 vs 0.660",
     "Observed, scoped to this frame: at the top of the queue the rule leads (P@50 0.880 vs 0.660); "
     "over the whole list the forest leads the fair rule reference (0.654 vs 0.609). "
     "Neither result is a general 'model beats rule' claim."),

    ("Week 5, section 3",
     "the rule is the best filter we have and still not a ranker",
     "'best' means among the methods I compared, on one month of data; 'not a ranker' is supported "
     "by the 0.505 unaided AUC against a 0.500 no-skill line",
     "Among the methods compared on this frame, the rule produced the highest measured precision at "
     "the top of the queue (P@50 0.880), while its unaided whole-list ROC-AUC (0.505) sits near the "
     "0.500 no-skill line - a filter, not a ranker, in this data."),
]

for i, (where, bold, evidence, safe) in enumerate(CLAIMS, 1):
    print("=" * 80)
    print(f"CLAIM {i}  ({where})")
    print(f"  bold:   {bold}")
    print(f"  bears:  {evidence}")
    print(f"  safe:   {safe}")

# ---- (b) Banned-phrasing scan over this notebook's own markdown -------------
import re
BANNED = ["proves", "proven", "causes", "caused", "will increase", "will improve",
          "will recover", "guarantee", "definitive", "undeniable", "no doubt",
          "predicts google", "algorithm rewards", "surely", "obviously"]

nb_path = Path("work/notebooks/w06_validation_audit.ipynb")
this_nb = json.loads(nb_path.read_text(encoding="utf-8"))
hits = []
for idx, cell in enumerate(this_nb["cells"]):
    if cell["cell_type"] != "markdown":
        continue  # the banned list itself lives in a code cell, so it cannot trip the scan
    # word-boundary match, so the paper's quoted "Improves the odds" (page 6)
    # is not a false positive on the substring "proves"
    for line in "".join(cell["source"]).splitlines():
        for phrase in BANNED:
            if re.search(r"\b" + re.escape(phrase) + r"\b", line.lower()):
                hits.append((idx, phrase, line.strip()[:200]))

print("\n" + "=" * 80)
print(f"Banned-phrasing scan over this notebook's markdown ({len(BANNED)} phrases) "
      f"-> {len(hits)} hit(s):")
for idx, phrase, line in hits:
    print(f"  cell {idx}: [{phrase}] {line}")
if not hits:
    print("  none - every claim sentence in this notebook stays inside the ladder "
          "(observed / measured / directional / decision-support).")

CLAIM 1  (Week 5, section 3, final line)
  bold:   the forest is a modest, genuine ranking improvement over what impressions alone can do
  bears:  one month of data, one portfolio, client-grouped OOF, AUC 0.654 vs 0.609 - a measured gap of +0.045
  safe:   On this March 2026 frame, under client-grouped out-of-fold scoring, the forest was measured at ROC-AUC 0.654 vs 0.609 for the impressions-only rank - an observed +0.045. Directional, decision-support: it reorders a review queue; it does not predict Google.
CLAIM 2  (Week 5, section 3)
  bold:   Did the model beat the baseline? ... on the whole list against a fair version of the rule: yes.
  bears:  true only for whole-list metrics on this frame; at P@50 the rule still leads 0.880 vs 0.660
  safe:   Observed, scoped to this frame: at the top of the queue the rule leads (P@50 0.880 vs 0.660); over the whole list the forest leads the fair rule reference (0.654 vs 0.609). Neither result is a general 'model beats rule' claim.
CLAIM 3  (W

### What the rewrite changes

The bold versions are not lies — they are **summaries that dropped their scope**. Each rewrite does the same three things:

1. **Names the frame** — "on this March 2026 frame", "among the methods compared" — because every number here is one month, one portfolio, pseudonymized data.
2. **Uses a ladder word** — observed, measured, directional — instead of a verdict word. Nothing in this notebook ran a controlled experiment, so no sentence gets to say *causes*, *proves*, or *will*.
3. **Ends as decision-support** — "these pages look worth reviewing first", "it reorders a review queue" — the form a reader can act on without being told a cross-sectional pattern is a lever.

**Scan result: 2 hits, both self-referential.** Both sit in the ladder-word rule under "What the rewrite changes" above — the one sentence in this notebook that *names* the words it refuses to use. Triage: mentions, not claims, and I am keeping them visible on purpose, because a scanner that never fires would be as untrustworthy as the leak test in section 3 that *did* fire. The three rewritten claims above contain none of them; the word-boundary match also keeps the paper's own quoted "Improves the odds" (page 6) from false-positiving on the substring.

**The habit this section exists to build:** every conclusion sentence, read alone and out of context, must be carryable by the table printed nearest to it — frame named, ladder word used, decision-support ending.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] Two paper findings named, with the methodology question for each, framed constructively (section 1)
- [x] Week-5 model re-run under a grouped split with a before/after comparison — both numbers shown, base rate under every table (section 2)
- [x] Leakage audit on the final feature set: label-derived assertions, a real future-window confession demo that was then removed, decision-derived check (section 3)
- [x] Real failure examples printed — concrete rows, not just metrics (section 2)
- [x] My boldest claims rewritten in safe language, plus an automated scan of this notebook's markdown (section 4)
- [x] Every number in the prose matches the output directly above/below it (re-checked after the final run)
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (IDs are pseudonyms; the paper path is a repo file)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.